## Step 1 — Install Dependencies

In [ ]:
!pip install nltk matplotlib tqdm -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✅ Dependencies ready')

In [ ]:
import os
import zipfile
from google.colab import files

# ── Option A: Upload zip files ──────────────────────────────────────────────
print('Upload your Flickr8k zip files...')
uploaded = files.upload()

os.makedirs('/content/flickr8k/images', exist_ok=True)
os.makedirs('/content/flickr8k/text',   exist_ok=True)

for fname in uploaded:
    print(f'Extracting {fname}...')
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/flickr8k/')

print('\n✅ Dataset extracted')
print('Files found:', os.listdir('/content/flickr8k/'))

In [ ]:
# ── Locate key paths (adjust if your structure differs) ─────────────────────
import glob

# Find captions file
token_files = glob.glob('/content/flickr8k/**/*.token.txt', recursive=True) + \
              glob.glob('/content/flickr8k/**/Flickr8k.token.txt', recursive=True) + \
              glob.glob('/content/flickr8k/**/*.token', recursive=True)

# Find images directory
img_dirs = glob.glob('/content/flickr8k/**/Flicker8k_Dataset', recursive=True) + \
           glob.glob('/content/flickr8k/**/Images', recursive=True) + \
           glob.glob('/content/flickr8k/**/*.jpg', recursive=True)

CAPTIONS_FILE = token_files[0] if token_files else '/content/flickr8k/Flickr8k.token.txt'
IMAGES_DIR    = os.path.dirname(img_dirs[0]) if img_dirs and img_dirs[0].endswith('.jpg') \
                else (img_dirs[0] if img_dirs else '/content/flickr8k/Flicker8k_Dataset')

print(f'Captions file : {CAPTIONS_FILE}')
print(f'Images folder : {IMAGES_DIR}')
print(f'Exists        : captions={os.path.exists(CAPTIONS_FILE)}, images={os.path.exists(IMAGES_DIR)}')

## Step 3 — Load & Preprocess Captions

In [ ]:
import re
from collections import Counter

# ── Parse captions file ─────────────────────────────────────────────────────
def load_captions(filepath):
    """Returns dict: {image_name: [cap1, cap2, ...]}."""
    captions = {}
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if '\t' in line:
                parts = line.split('\t', 1)
                img   = parts[0].split('#')[0].strip()
                cap   = parts[1].strip()
            elif ',' in line:
                parts = line.split(',', 1)
                img, cap = parts[0].strip(), parts[1].strip()
            else:
                continue
            captions.setdefault(img, []).append(cap)
    return captions

def clean_caption(caption):
    caption = caption.lower()
    caption = re.sub(r"[^a-z\s]", "", caption)
    caption = re.sub(r"\s+", " ", caption).strip()
    return 'startseq ' + caption + ' endseq'

raw_captions = load_captions(CAPTIONS_FILE)
print(f'Total images with captions: {len(raw_captions)}')
print(f'Captions per image        : {len(list(raw_captions.values())[0])}')

# Clean
clean_captions = {}
for img, caps in raw_captions.items():
    clean_captions[img] = [clean_caption(c) for c in caps]

# Sample
sample_img = list(clean_captions.keys())[0]
print(f'\nSample ({sample_img}):')
for c in clean_captions[sample_img]:
    print(' ', c)

In [ ]:
# ── Build vocabulary ────────────────────────────────────────────────────────
MIN_WORD_FREQ = 5  # ignore words appearing fewer than this many times

all_words = []
for caps in clean_captions.values():
    for cap in caps:
        all_words.extend(cap.split())

freq = Counter(all_words)
vocab = [w for w, c in freq.items() if c >= MIN_WORD_FREQ]
vocab = sorted(vocab)

# Word ↔ index mappings
word2idx = {w: i+1 for i, w in enumerate(vocab)}  # 0 = padding
idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(word2idx) + 1
MAX_LEN    = max(len(cap.split()) for caps in clean_captions.values() for cap in caps)

print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'Max caption len : {MAX_LEN}')

## Step 4 — Train / Val / Test Split

In [ ]:
import random

# Filter to images that actually exist on disk
all_images = [
    img for img in clean_captions
    if os.path.exists(os.path.join(IMAGES_DIR, img))
]
random.seed(42)
random.shuffle(all_images)

n       = len(all_images)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)

train_imgs = all_images[:n_train]
val_imgs   = all_images[n_train:n_train+n_val]
test_imgs  = all_images[n_train+n_val:]

print(f'Train : {len(train_imgs)}')
print(f'Val   : {len(val_imgs)}')
print(f'Test  : {len(test_imgs)}')

## Step 5 — Extract CNN Features (ResNet-50)

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from tqdm import tqdm
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# ResNet-50 as feature extractor (remove final FC layer)
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet = nn.Sequential(*list(resnet.children())[:-1])  # output: (batch, 2048, 1, 1)
resnet = resnet.to(DEVICE)
resnet.eval()

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

def extract_features(img_list, images_dir, batch_size=64):
    features = {}
    for i in tqdm(range(0, len(img_list), batch_size), desc='Extracting features'):
        batch_names = img_list[i:i+batch_size]
        batch_tensors = []
        valid_names = []
        for name in batch_names:
            path = os.path.join(images_dir, name)
            try:
                img = Image.open(path).convert('RGB')
                batch_tensors.append(transform(img))
                valid_names.append(name)
            except Exception:
                pass
        if not batch_tensors:
            continue
        batch = torch.stack(batch_tensors).to(DEVICE)
        with torch.no_grad():
            feats = resnet(batch).squeeze(-1).squeeze(-1)  # (B, 2048)
        for name, feat in zip(valid_names, feats):
            features[name] = feat.cpu().numpy()
    return features

print('Extracting features for all images (this takes a few minutes)...')
all_imgs_list = list(set(train_imgs + val_imgs + test_imgs))
img_features  = extract_features(all_imgs_list, IMAGES_DIR)

print(f'\n✅ Features extracted for {len(img_features)} images')
print(f'Feature vector size: {list(img_features.values())[0].shape}')

## Step 6 — Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CaptionDataset(Dataset):
    def __init__(self, img_list, captions_dict, features_dict, word2idx, max_len):
        self.data = []
        for img in img_list:
            if img not in features_dict:
                continue
            feat = features_dict[img]
            for cap in captions_dict.get(img, []):
                tokens = cap.split()
                # Build input / target pairs using teacher forcing
                # Input seq: startseq w1 w2 ... wN
                # Target   : w1 w2 ... wN endseq
                seq = [word2idx.get(w, 0) for w in tokens]
                for t in range(1, len(seq)):
                    in_seq  = seq[:t]
                    out_word = seq[t]
                    # Pad input
                    in_seq = in_seq + [0] * (max_len - len(in_seq))
                    self.data.append((feat, in_seq[:max_len], out_word))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        feat, in_seq, out_word = self.data[idx]
        return (
            torch.tensor(feat,     dtype=torch.float32),
            torch.tensor(in_seq,   dtype=torch.long),
            torch.tensor(out_word, dtype=torch.long),
        )

train_dataset = CaptionDataset(train_imgs, clean_captions, img_features, word2idx, MAX_LEN)
val_dataset   = CaptionDataset(val_imgs,   clean_captions, img_features, word2idx, MAX_LEN)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples : {len(train_dataset):,}')
print(f'Val samples   : {len(val_dataset):,}')

## Step 7 — CNN + LSTM Model

In [ ]:
class CaptionModel(nn.Module):
    """
    Architecture:
      Image feature  (2048) ──► FC ──► embed_dim
      Caption tokens (seq)  ──► Embedding ──► embed_dim
      Both are added and fed into LSTM ──► FC ──► vocab_size
    """
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2, dropout=0.5):
        super().__init__()
        self.img_fc   = nn.Linear(2048, embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout   = nn.Dropout(dropout)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, num_layers,
                                 batch_first=True, dropout=dropout)
        self.fc_out    = nn.Linear(hidden_dim, vocab_size)

    def forward(self, img_feat, seq):
        # img_feat : (B, 2048)
        # seq      : (B, max_len)  — padded token indices
        img_embed = self.dropout(torch.relu(self.img_fc(img_feat)))  # (B, embed)
        seq_embed = self.dropout(self.embedding(seq))                # (B, T, embed)

        # Add image embedding to every time step
        img_embed = img_embed.unsqueeze(1).expand_as(seq_embed)      # (B, T, embed)
        combined  = seq_embed + img_embed                            # (B, T, embed)

        lstm_out, _ = self.lstm(combined)                            # (B, T, hidden)
        out = self.fc_out(lstm_out[:, -1, :])                        # (B, vocab_size)
        return out


EMBED_DIM  = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT    = 0.5

model = CaptionModel(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

## Step 8 — Training

In [ ]:
import matplotlib.pyplot as plt

EPOCHS    = 20
LR        = 1e-3
CLIP_GRAD = 5.0

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5, verbose=True)
criterion = nn.CrossEntropyLoss(ignore_index=0)

train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0
    for feat, in_seq, target in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [train]', leave=False):
        feat, in_seq, target = feat.to(DEVICE), in_seq.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        logits = model(feat, in_seq)
        loss   = criterion(logits, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        optimizer.step()
        total_loss += loss.item()

    avg_train = total_loss / len(train_loader)
    train_losses.append(avg_train)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for feat, in_seq, target in val_loader:
            feat, in_seq, target = feat.to(DEVICE), in_seq.to(DEVICE), target.to(DEVICE)
            logits = model(feat, in_seq)
            val_loss += criterion(logits, target).item()

    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)
    scheduler.step(avg_val)

    print(f'Epoch {epoch:02d}/{EPOCHS}  |  Train Loss: {avg_train:.4f}  |  Val Loss: {avg_val:.4f}')

    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), '/content/best_caption_model.pth')
        print(f'  ✅ Best model saved (val loss: {best_val_loss:.4f})')

# ── Plot losses ────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses,   label='Val Loss',   marker='s')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/loss_curve.png', dpi=150)
plt.show()
print('Training complete!')

## Step 9 — Caption Generation (Greedy & Beam Search)

In [ ]:
# Load best model
model.load_state_dict(torch.load('/content/best_caption_model.pth', map_location=DEVICE))
model.eval()
print('✅ Best model loaded')


def get_image_feature(img_path):
    img = Image.open(img_path).convert('RGB')
    t   = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = resnet(t).squeeze(-1).squeeze(-1)
    return feat


def generate_caption_greedy(img_path, max_len=MAX_LEN):
    """Greedy decoding — picks highest probability word at each step."""
    feat   = get_image_feature(img_path)
    in_seq = [word2idx.get('startseq', 1)]

    result = []
    for _ in range(max_len):
        padded = in_seq + [0] * (MAX_LEN - len(in_seq))
        seq_t  = torch.tensor([padded[:MAX_LEN]], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            logits   = model(feat, seq_t)
            next_idx = logits.argmax(dim=-1).item()
        word = idx2word.get(next_idx, '')
        if word == 'endseq' or next_idx == 0:
            break
        result.append(word)
        in_seq.append(next_idx)

    return ' '.join(result)


def generate_caption_beam(img_path, beam_width=3, max_len=MAX_LEN):
    """Beam search — keeps top-k candidates at each step for better captions."""
    feat = get_image_feature(img_path)
    start_idx = word2idx.get('startseq', 1)
    end_idx   = word2idx.get('endseq', 2)

    # Each beam: (score, sequence)
    beams     = [(0.0, [start_idx])]
    completed = []

    for _ in range(max_len):
        candidates = []
        for score, seq in beams:
            if seq[-1] == end_idx:
                completed.append((score, seq))
                continue
            padded = seq + [0] * (MAX_LEN - len(seq))
            seq_t  = torch.tensor([padded[:MAX_LEN]], dtype=torch.long).to(DEVICE)
            with torch.no_grad():
                logits = model(feat, seq_t)
                log_probs = torch.log_softmax(logits, dim=-1).squeeze()
            top_probs, top_idxs = log_probs.topk(beam_width)
            for prob, idx in zip(top_probs.tolist(), top_idxs.tolist()):
                candidates.append((score + prob, seq + [idx]))

        if not candidates:
            break
        beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]

    completed += beams
    best_seq = max(completed, key=lambda x: x[0] / len(x[1]))[1]

    words = [idx2word.get(i, '') for i in best_seq
             if i not in (start_idx, end_idx, 0)]
    return ' '.join(words)


print('Caption generation functions ready!')

## Step 10 — Test on Sample Images

In [ ]:
import random
import matplotlib.pyplot as plt

# Pick 6 random test images
samples = random.sample([img for img in test_imgs if img in img_features], min(6, len(test_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, img_name in zip(axes, samples):
    img_path = os.path.join(IMAGES_DIR, img_name)
    img      = Image.open(img_path).convert('RGB')

    greedy_cap = generate_caption_greedy(img_path)
    beam_cap   = generate_caption_beam(img_path, beam_width=3)

    ax.imshow(img)
    ax.axis('off')
    ax.set_title(
        f'Greedy: {greedy_cap}\nBeam:   {beam_cap}',
        fontsize=8, wrap=True, loc='left'
    )

plt.suptitle('Generated Captions on Test Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/test_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11 — BLEU Score Evaluation on Test Set

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

smoother = SmoothingFunction().method1

references_corpus  = []  # list of list of reference token lists
hypotheses_greedy  = []  # list of hypothesis token lists (greedy)
hypotheses_beam    = []  # list of hypothesis token lists (beam)

eval_imgs = [img for img in test_imgs if img in img_features][:500]  # cap at 500 for speed

print(f'Evaluating on {len(eval_imgs)} test images...')

for img_name in tqdm(eval_imgs, desc='Evaluating'):
    img_path = os.path.join(IMAGES_DIR, img_name)
    refs     = [word_tokenize(c.replace('startseq', '').replace('endseq', '').strip())
                for c in clean_captions.get(img_name, [])]
    if not refs:
        continue

    greedy = generate_caption_greedy(img_path)
    beam   = generate_caption_beam(img_path, beam_width=3)

    references_corpus.append(refs)
    hypotheses_greedy.append(word_tokenize(greedy))
    hypotheses_beam.append(word_tokenize(beam))

# Corpus BLEU
b1_g = corpus_bleu(references_corpus, hypotheses_greedy, weights=(1,0,0,0))
b2_g = corpus_bleu(references_corpus, hypotheses_greedy, weights=(.5,.5,0,0))
b3_g = corpus_bleu(references_corpus, hypotheses_greedy, weights=(.33,.33,.33,0))
b4_g = corpus_bleu(references_corpus, hypotheses_greedy, weights=(.25,.25,.25,.25))

b1_b = corpus_bleu(references_corpus, hypotheses_beam, weights=(1,0,0,0))
b2_b = corpus_bleu(references_corpus, hypotheses_beam, weights=(.5,.5,0,0))
b3_b = corpus_bleu(references_corpus, hypotheses_beam, weights=(.33,.33,.33,0))
b4_b = corpus_bleu(references_corpus, hypotheses_beam, weights=(.25,.25,.25,.25))

print('\n' + '='*50)
print(f'{"Metric":<10} {"Greedy":>10} {"Beam (k=3)":>12}')
print('='*50)
for label, g, b in [('BLEU-1', b1_g, b1_b), ('BLEU-2', b2_g, b2_b),
                     ('BLEU-3', b3_g, b3_b), ('BLEU-4', b4_g, b4_b)]:
    print(f'{label:<10} {g:>10.4f} {b:>12.4f}')
print('='*50)

# Verdict
def verdict(score):
    if   score >= 0.40: return 'Excellent ✅'
    elif score >= 0.30: return 'Good ✅'
    elif score >= 0.20: return 'Moderate ⚠️'
    elif score >= 0.10: return 'Low ❌'
    else:               return 'Very Low ❌'

print(f'\nGreedy BLEU-4 → {verdict(b4_g)}')
print(f'Beam   BLEU-4 → {verdict(b4_b)}')

## Step 12 — Test on Your Own Image

In [ ]:
from google.colab import files as colab_files

print('Upload your own image...')
uploaded = colab_files.upload()

for fname in uploaded:
    img_path = f'/content/{fname}'
    img      = Image.open(img_path).convert('RGB')

    greedy_cap = generate_caption_greedy(img_path)
    beam_cap   = generate_caption_beam(img_path, beam_width=3)

    plt.figure(figsize=(7, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Greedy: {greedy_cap}\nBeam (k=3): {beam_cap}', fontsize=10, loc='left')
    plt.tight_layout()
    plt.show()

    print(f'\nImage       : {fname}')
    print(f'Greedy      : {greedy_cap}')
    print(f'Beam (k=3)  : {beam_cap}')

## Step 13 — Save & Download Model

In [ ]:
import json

# Save vocabulary
with open('/content/vocab.json', 'w') as f:
    json.dump({'word2idx': word2idx, 'idx2word': {str(k): v for k,v in idx2word.items()},
               'vocab_size': VOCAB_SIZE, 'max_len': MAX_LEN}, f)

print('Files saved:')
print('  /content/best_caption_model.pth  — model weights')
print('  /content/vocab.json              — vocabulary')
print('  /content/loss_curve.png          — training curve')
print('  /content/test_predictions.png    — sample predictions')

# Download everything
for fpath in ['/content/best_caption_model.pth', '/content/vocab.json',
              '/content/loss_curve.png', '/content/test_predictions.png']:
    if os.path.exists(fpath):
        colab_files.download(fpath)

print('\n✅ All files downloaded!')